In [12]:
import os
import re

import dotenv
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone import Pinecone, ServerlessSpec
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_pinecone import PineconeVectorStore
from langchain.chains import create_retrieval_chain
from langchain.prompts import ChatPromptTemplate

dotenv.load_dotenv()

DOCS_CHUNK_SIZE = int(os.getenv("DOCS_CHUNK_SIZE"))
DOCS_CHUNK_OVERLAP = int(os.getenv("DOCS_CHUNK_OVERLAP"))
ALUMNO = os.getenv("ALUMNO")


Leer CVs 

In [38]:
class Agent:
    def __init__(self, filename:str, filepath: str, embeddings, llm):
        name = os.path.splitext(filename)[0].lower()
        self.name = name
        self.cv_filepath = filepath
        self.embeddings = embeddings
        self.index = name
        self.llm = llm
        self.student = name == ALUMNO

        self.cv = None
        self.splits = None
        self.vectorstore = None
        self.retriever = None
        self.chain = None

        self._load_cv()
        self._create_vectorstore()
        self._create_chain()

    def _load_cv(self):
        loader = PyPDFLoader(self.cv_filepath)
        self.cv = loader.load()

        splitter = RecursiveCharacterTextSplitter(chunk_size=DOCS_CHUNK_SIZE, chunk_overlap=DOCS_CHUNK_OVERLAP)
        self.chunks = splitter.split_documents(self.cv)
        self.vectors = embeddings.embed_documents([doc.page_content for doc in self.chunks])

    def _create_vectorstore(self):
        self.vectorstore = PineconeVectorStore.from_documents(
            self.chunks,
            self.embeddings,
            index_name=self.index
        )
        self.retriever = self.vectorstore.as_retriever()

    def _create_chain(self):
        prompt = ChatPromptTemplate.from_template(
            "Usa el siguiente CV de {name} para responder la pregunta.\n"
            "Contexto:\n{context}\n\n"
            "Pregunta: {input}"
        )

        doc_chain = create_stuff_documents_chain(self.llm, prompt)
        self.chain = create_retrieval_chain(self.retriever, doc_chain)

    def answer(self, question: str, history) -> str:
        result = self.chain.invoke({"input": question, "name": self.name, "chat_history": history})
        return result["answer"]

In [39]:
embeddings = HuggingFaceEmbeddings(model_name=os.getenv("EMBEDDINGS_MODEL"))
groq = ChatGroq(model=os.getenv("GROQ_MODEL"), temperature=0)

In [40]:
def create_agents_from_cvs(directory="docs"):
    agents = []
    for filename in os.listdir(directory):
        if filename.endswith(".pdf"):
            filepath = os.path.join(directory, filename)
            agent = Agent(filename, filepath, embeddings=embeddings, llm=groq)
            agents.append(agent)

    return agents

In [41]:
agents = create_agents_from_cvs("docs")

Los secciono en chunks de datos

In [17]:
pinecone = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
spec = ServerlessSpec(cloud=os.getenv("PINECONE_CLOUD"), region=os.getenv("PINECONE_REGION"))

Configurar indices

In [18]:
def recreate_index(index_name, pinecone, spec):
    if index_name in pinecone.list_indexes().names():
        pinecone.delete_index(index_name)
    print("index {} borrado".format(index_name))

    if index_name not in pinecone.list_indexes().names():
        print("index creado con el nombre: {}".format(index_name))
        pinecone.create_index(
            index_name,
            dimension=384,
            metric='cosine',
            spec=spec
        )
    else:
        print("el index con el nombre {} ya estaba creado".format(index_name))

Subir los vectores a Pinecone

In [23]:
def upload_to_pinecone(index, documents, vectors):
    for i, (doc, vector) in enumerate(zip(documents, vectors)):
        pinecone_index = pinecone.Index(index)
        pinecone_index.upsert([
            (
                f"chunk-{i}",
                vector,
                {"text": doc.page_content}
            )
        ])
    
    print(f"vectores cargados en {index}")

In [24]:
for agent in agents:
    recreate_index(agent.index, pinecone, spec)
    upload_to_pinecone(agent.index, agent.chunks, agent.vectors)

index christian borrado
index creado con el nombre: christian
vectores cargados en christian
index joaquin borrado
index creado con el nombre: joaquin
vectores cargados en joaquin
index javier borrado
index creado con el nombre: javier
vectores cargados en javier


Utilizaré Groq como LLM para responder preguntas sobre los CVs. 

Creo un agente por cada uno de los curriculums cargados

In [35]:
def select_agents(question: str, agents):
    selected = []

    for agent in agents:
        if re.search(rf"\b{agent.name}\b", question, re.IGNORECASE):
            selected.append(agent)

    if not selected:
        selected = [agent for agent in agents if agent.student]

    return selected

In [36]:
def answer_question(question: str, agents, conversation_history=[]):
    selected_agents = select_agents(question, agents)
    answers = []

    for agent in selected_agents:
        result = agent.answer(question, conversation_history)
        answers.append({
            "agent": agent.name,
            "answer": result,
        })

    final_answer = "\n\n".join(
        [f"[{a['agent'].capitalize()}]: {a['answer']}" for a in answers]
    )
    print(final_answer)


In [33]:
conversation_history = []

In [45]:
answer_question("¿Qué experiencia tiene Christian en IA?", agents, conversation_history)

('[Christian]: Según el CV de Christian, tiene experiencia en el desarrollo de aplicaciones de Data Mining utilizando tecnologías como MySQL, Selenium Web Driver, Fiddler y otras herramientas de crawling web. Esto sugiere que tiene experiencia en el análisis y procesamiento de grandes conjuntos de datos, lo que es una habilidad relevante en el campo de la Inteligencia Artificial (IA).\n\nAdemás, su experiencia en el desarrollo de una Data Warehouse en MySQL utilizando Pentaho Data Integration como herramienta ETL (Extract, Transform, Load) también sugiere que tiene habilidades en la integración y análisis de datos, lo que es fundamental en la IA.\n\nSin embargo, no hay menciones explícitas de experiencia en áreas específicas de la IA como el aprendizaje automático, el procesamiento del lenguaje natural o la visión artificial. Por lo tanto, se puede concluir que Christian tiene experiencia en áreas relacionadas con la IA, como el Data Mining y el análisis de datos, pero no necesariament

In [42]:
answer_question("¿Qué experiencia tiene en IA?", agents, conversation_history)


[Christian]: Según el CV de Christian, tiene experiencia en Data Mining, que es un campo relacionado con la Inteligencia Artificial (IA). En su experiencia laboral en Navent (08/2015 - Presente), se menciona que:

* Diseña y desarrolla aplicaciones de Data Mining en C# utilizando MySQL, Selenium Web Driver, Fiddler y otras tecnologías de crawling web.
* Desarrolla un Data Warehouse en MySQL utilizando Pentaho Data Integration como herramienta ETL.

Además, en su sección de habilidades y conocimientos técnicos, se menciona que tiene experiencia en:

* Programación en Java, C#, Python, C, Javascript, PL/SQL, Smalltalk
* Bases de datos: MySQL, Oracle 11g, SQLServer, Cassandra, SQLite
* Sistemas operativos: Linux, iOS, Android, Windows
* Herramientas: Visual Studio, Eclipse, CSS, Selenium WebDriver, Pentaho, ZK framework, SOAPUI, REST, UML, WinForms, JUnit, HTML, GIT

Sin embargo, no se menciona explícitamente experiencia en IA en general, sino más bien en Data Mining, que es un subcampo d

In [43]:
answer_question("¿Qué experiencia tiene Javier en IA?", agents, conversation_history)


[Javier]: Según el CV proporcionado, no hay menciones a la experiencia de Javier en Inteligencia Artificial (IA). El CV se enfoca en su formación y experiencia en el campo de la música, específicamente en el contrabajo y la enseñanza de instrumentos musicales. No se menciona ninguna relación con la Inteligencia Artificial.


In [48]:
answer_question("¿Hay algo en común entre Javier y Joaquín?", agents, conversation_history)


('[Javier]: No hay información suficiente en el CV de Javier para determinar si hay algo en común entre él y Joaquín, ya que no se proporciona información sobre Joaquín. El CV solo describe la formación académica, experiencia laboral y habilidades de Javier. Para determinar si hay algo en común, necesitaríamos información adicional sobre Joaquín.',
 [{'agent': 'javier',
   'answer': 'No hay información suficiente en el CV de Javier para determinar si hay algo en común entre él y Joaquín, ya que no se proporciona información sobre Joaquín. El CV solo describe la formación académica, experiencia laboral y habilidades de Javier. Para determinar si hay algo en común, necesitaríamos información adicional sobre Joaquín.'}])